In [898]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [899]:
np.random.seed(42)
n_rows=500

area = np.random.uniform(500,500,n_rows)
age = np.random.randint(0,100,n_rows).astype(float)
distance = np.random.uniform(1,50,n_rows)
rooms = np.random.randint(1,10,n_rows)
amenities = np.random.uniform(1,10,n_rows)

prop_types = ['Apartment','House','Will']
property_type = np.random.choice(prop_types,n_rows)

energy_levels = ['low','Medium','High']
energy_rating = np.random.choice(energy_levels,n_rows)

regions = ['North','South','East','West']
region = np.random.choice(regions,n_rows)

noise = np.random.normal(0,15000,n_rows)
price = 10000 + (50*area) + (0.05 * (area**2)) - (500*age) + (3000 * rooms) + noise

df=pd.DataFrame({
    'Area_sqft' : area,
    'Age_years' : age,
    'Distance_to_City' : distance,
    'Rooms' : rooms,
    'Amenities_score':amenities,
    'Property_type' : property_type,
    'Energy_Rating':energy_rating,
    'Region' : region,
    'Rennovated': np.random.choice([0,1],n_rows),
    'Tax_Rate' : np.random.uniform(0.05,0.15,n_rows),
    'Lot_Size' : area * np.random.uniform(1.1,2.0),
    'Price' : price
})

df.loc[df.sample(frac=0.1).index,'Age_years'] = np.nan

outlier_indices = df.sample(frac=0.005).index
df.loc[outlier_indices,'Price'] = df.loc[outlier_indices,'Price']*10
df.loc[outlier_indices,'Area_sqft'] = df.loc[outlier_indices,'Area_sqft']*5

df.to_csv('polynomial_data_with_outlier.csv',index=False)
print("Dataset created : 50,000 rows , 12 columns")

Dataset created : 50,000 rows , 12 columns


In [900]:
data=pd.read_csv('polynomial_data_with_outlier.csv')

In [901]:
data.head()

,Area_sqft,Age_years,Distance_to_City,Rooms,Amenities_score,Property_type,Energy_Rating,Region,Rennovated,Tax_Rate,Lot_Size,Price
0,500.0,62.0,37.468233,2,8.113040,Apartment,low,East,1,0.053002,723.129659,15402.448940
1,500.0,16.0,11.073411,6,6.239334,Will,low,East,0,0.094453,723.129659,57041.447549
2,500.0,72.0,39.601705,7,7.128773,Apartment,low,West,0,0.100462,723.129659,22285.733863
3,500.0,32.0,30.582355,9,2.707254,Apartment,High,East,0,0.061728,723.129659,44973.276199
4,500.0,NaN,6.599098,3,6.570534,Apartment,low,North,1,0.066906,723.129659,26059.369748


In [902]:
data.isna().sum()

Area_sqft            0
Age_years           50
Distance_to_City     0
Rooms                0
Amenities_score      0
Property_type        0
Energy_Rating        0
Region               0
Rennovated           0
Tax_Rate             0
Lot_Size             0
Price                0
dtype: int64

In [903]:
data['Age_years'] = data['Age_years'].fillna(df['Age_years'].mean())

In [904]:
data.isna().sum()

Area_sqft           0
Age_years           0
Distance_to_City    0
Rooms               0
Amenities_score     0
Property_type       0
Energy_Rating       0
Region              0
Rennovated          0
Tax_Rate            0
Lot_Size            0
Price               0
dtype: int64

In [905]:
numerical_features = [feature for feature in data.columns if data[feature].dtype!='O']
categorical_features = [feature for feature in data.columns if data[feature].dtype=='O']

print(numerical_features)
print(categorical_features)

['Area_sqft', 'Age_years', 'Distance_to_City', 'Rooms', 'Amenities_score', 'Rennovated', 'Tax_Rate', 'Lot_Size', 'Price']
['Property_type', 'Energy_Rating', 'Region']


In [906]:
data.shape

(500, 12)

In [907]:
for col in data.select_dtypes(include=['int64','float64']).columns:
    Q1=data[col].quantile(0.25)
    Q2=data[col].median()
    Q3=data[col].quantile(0.75)

    IQR = Q3-Q1
    lower_fence = Q1-(1.5*IQR)
    higher_fence = Q3+(1.5*IQR)

    data=data[(data[col]>=lower_fence) & (data[col]<=higher_fence)]

In [908]:
from sklearn.preprocessing import StandardScaler,OneHotEncoder,OrdinalEncoder
from sklearn.compose import ColumnTransformer

Oe=OrdinalEncoder()
data['Energy_Rating']=Oe.fit_transform(data[['Energy_Rating']]).astype(int)

In [909]:
data.head()

,Area_sqft,Age_years,Distance_to_City,Rooms,Amenities_score,Property_type,Energy_Rating,Region,Rennovated,Tax_Rate,Lot_Size,Price
0,500.0,62.000000,37.468233,2,8.113040,Apartment,2,East,1,0.053002,723.129659,15402.448940
1,500.0,16.000000,11.073411,6,6.239334,Will,2,East,0,0.094453,723.129659,57041.447549
2,500.0,72.000000,39.601705,7,7.128773,Apartment,2,West,0,0.100462,723.129659,22285.733863
3,500.0,32.000000,30.582355,9,2.707254,Apartment,0,East,0,0.061728,723.129659,44973.276199
4,500.0,48.137778,6.599098,3,6.570534,Apartment,2,North,1,0.066906,723.129659,26059.369748


In [910]:
data.shape

(497, 12)

In [911]:
X = data.drop('Price',axis=1)
y = data['Price']

In [912]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=42)

In [913]:
num_features = X.select_dtypes(exclude=object).columns
cat_features = X.select_dtypes(include=object).columns

scaler=StandardScaler()
ohe=OneHotEncoder(drop="first")

preprocessor = ColumnTransformer([
    ("Ohe",ohe,cat_features),
    ("Scaler",scaler,num_features)
])

In [914]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [915]:
X_train_df = pd.DataFrame(X_train,columns=preprocessor.get_feature_names_out())
X_train_df.head()

,Ohe__Property_type_House,Ohe__Property_type_Will,Ohe__Region_North,Ohe__Region_South,Ohe__Region_West,Scaler__Area_sqft,Scaler__Age_years,Scaler__Distance_to_City,Scaler__Rooms,Scaler__Amenities_score,Scaler__Energy_Rating,Scaler__Rennovated,Scaler__Tax_Rate,Scaler__Lot_Size
0,1.0,0.0,0.0,0.0,0.0,0.0,-1.590735,0.848341,0.854108,-1.508171,-0.088353,-0.978721,-0.086817,0.0
1,1.0,0.0,0.0,0.0,1.0,0.0,0.001567,-0.398212,-0.335885,-0.722589,-0.088353,-0.978721,0.078257,0.0
2,1.0,0.0,0.0,0.0,1.0,0.0,1.151019,-1.295914,-0.732550,-1.057717,1.128953,-0.978721,1.405965,0.0
3,0.0,0.0,0.0,0.0,1.0,0.0,0.718111,0.623560,0.457444,-0.504441,-1.305659,-0.978721,-1.201049,0.0
4,0.0,1.0,0.0,0.0,1.0,0.0,-1.626811,-1.336346,-1.129214,1.322542,-0.088353,1.021742,1.480650,0.0


In [916]:
from sklearn.preprocessing import PolynomialFeatures

In [917]:
from sklearn.linear_model import LinearRegression,LassoCV
lr=LinearRegression()
lr.fit(X_train,y_train)
y_pred = lr.predict(X_test)

In [918]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
print(r2_score(y_test,y_pred))
print(mean_absolute_error(y_test,y_pred))
print(mean_squared_error(y_test,y_pred))

0.5146029370250335
11815.599435359558
229907818.90194583


In [919]:
lasso=LassoCV(cv=5)
lasso.fit(X_train,y_train)
y_pred1 = lasso.predict(X_test)

print(r2_score(y_test,y_pred1))
print(mean_absolute_error(y_test,y_pred1))
print(mean_squared_error(y_test,y_pred1))


0.5222076176677354
11688.356276765602
226305869.74862018
